# std::unique
1. Why better than `new`?
- more safety (prevent hidden memory leak)

In [ ]:
void process(std::unique_ptr<Widget>(new Widget()), doSomethingElse());
// 编译器在生成机器码时，这一行代码通常包含三个步骤：
// 1. 执行 new Widget() 分配内存并构造对象。
// 2. 构造 std::unique_ptr<Widget> 对象。
// 3. 调用 doSomethingElse()。

由于求值顺序未定，编译器可能会为了优化将顺序调整为：1 → 3 → 2。
如果在步骤 1（new 成功）之后，执行步骤 3（doSomethingElse()）时抛出了异常，此时步骤 2 尚未执行，智能指针还没有接管这段内存。结果就是：刚刚 new 出来的 Widget 对象永远无法被释放，造成内存泄漏。

In [ ]:
void process(std::unique_ptr<Widget> ptr, void doSomethingElse());

因为 std::make_unique 是一个原子性的函数调用，它内部会保证 new 和智能指针的初始化在一个调用栈内完整完成，绝不会被外部的其他函数调用强行“插足”。  
注： 虽然 C++17 修改了求值顺序规则，使得这种特定的参数泄漏风险降低，但遵循 std::make_unique 的习惯依然是多线程和异常安全编程的最佳实践。

- Reduce Code Redundancy (DRY principle)

In [ ]:
// 传统方式：重复书写
std::unique_ptr<VeryLongClassNameWithTemplates<int>> ptr(new VeryLongClassNameWithTemplates<int>());

// 现代方式：结合 auto 极其清爽
auto ptr = std::make_unique<VeryLongClassNameWithTemplates<int>>();


- Safer Group Processing

In [ ]:
// 传统方式：需要确保类型声明带 []，一旦写错后续释放会崩溃
std::unique_ptr<int[]> arr(new int[5]); 

// 现代方式：语法极其直观，且自动推导正确的释放行为
auto arr = std::make_unique<int[]>(5); // 分配一个大小为 5 的 int 数组

2. When not to use std::make_unique  
- Customer Deleter (eg. fd, db connect, C API)
- Brace Initializer

In [ ]:
// 如果 Widget 有一个接收 std::initializer_list 的构造函数
// auto ptr = std::make_unique<Widget>({1, 2, 3}); // 编译报错！
std::unique_ptr<Widget> ptr(new Widget{1, 2, 3}); // 只能用传统方式


# Variable Template  
在 C++14 中引入的变量模板（Variable Templates），彻底补齐了 C++ 模板家族的最后一块拼图。在此之前，C++ 已经有了类模板（Class Templates）、函数模板（Function Templates）以及别名模板（Alias Templates），唯独缺少了变量的模板化能力。  
1. Why we need it?  
Before C++14, if we want to define a mathematical constants of generics (e.g. pi), and let it support float, double and long double, we can only save the country by two “curves”:

In [ ]:
// 做法 A：使用类模板的静态成员变量（极其臃肿）
template <typename T>
struct MyConstants {
    static constexpr T pi = T(3.1415926535897932385L);
};

In [ ]:
// 做法 B：使用函数模板返回常量（有运行时开销风险 / 语法不像变量）
template <typename T>
constexpr T pi() { return T(3.1415926535897932385L);

2. The perfect solution for C++14

In [ ]:
#include <iostream>
#include <iomanip>

using namespace std;

// 1. 定义一个变量模板
template<typename T>
constexpr T pi = T(3.1415926535897932384626433832795L);

// int main() {
    // 2. 根据不同的模板参数，直接生成并使用对应类型的变量
    float  pi_f  = pi<float>;       // 实例化为一个 float 变量
    double pi_d  = pi<double>;      // 实例化为一个 double 变量
    
    std::cout << std::setprecision(15);
    std::cout << "Float pi: " << pi_f << std::endl;  // 精度较低
    std::cout << "Double pi: " << pi_d << std::endl; // 精度较高
    
//     return 0;
// }


// Output:
// Float pi: 3.14159274101257
// Double pi: 3.14159265358979


3. 变量模板的非类型模板参数（NTTP）

变量模板不仅支持类型参数（typename T），同样也支持非类型模板参数。这在元编程和配置常量时非常有用。

In [ ]:
// 传入一个整数 N，自动生成一个大小为 N 的全 1 数组（结合 C++14 泛型能力）
template<int N>
const int repeated_ones[N] = { [0 ... N-1] = 1 }; // 注：某些编译器支持这种快速初始化，或用 std::array

// 更常见的做法：结合 std::array
#include <array>
template<std::size_t N>
const std::array<int, N> zero_vector{}; // 得到一个大小为 N 且全为 0 的 array

auto my_array = zero_vector<5>; // 得到一个 std::array<int, 5> 且全为 0



4. 标准库中的标准应用：_v 家族的诞生

你在现代 C++ 开发中，一定经常用到 std::is_same_v<A, B> 或 std::is_integral_v<T>。这些带有 _v 后缀的工具，底层全部都是通过 C++14 变量模板实现的！
在 C++11 中，要获取一个类型特征（Type Traits）的布尔值，你必须写 .value，代码非常冗长：

In [ ]:
// C++11 做法
bool result = std::is_same<int, T>::value;

而在 C++14 中，标准库利用变量模板为所有的类型特征做了一层精简的包装：

In [ ]:
// 别名/变量模板的底层实现原理大致如下：
template<typename T, typename U>
constexpr bool is_same_v = std::is_same<T, U>::value;

// C++14/17/20 现代做法
bool result = std::is_same_v<int, T>; // 清爽、直观



5. 变量模板的特化（Specialization）

和类模板一样，变量模板也支持全特化和偏特化。这使得你可以为特定的类型指定完全不同的行为或数值。

In [ ]:
#include <iostream>
#include <string>

// 基础变量模板
template<typename T>
std::string type_name = "Unknown Type";

// 全特化：针对 int 类型
template<>
std::string type_name<int> = "Integer";

// 全特化：针对 double 类型
template<>
std::string type_name<double> = "Floating Point (Double)";

int main() {
    std::cout << type_name<char> << std::endl;   // 输出: Unknown Type
    std::cout << type_name<int> << std::endl;    // 输出: Integer
    std::cout << type_name<double> << std::endl; // 输出: Floating Point (Double)
    return 0;
}


# std::integer_sequence  
std::integer_sequence（以及它的特化版本 std::index_sequence）是标准库中用于编译期元编程（Template Metaprogramming）的超级核心工具。它于 C++14 正式引入，专门用来表示一个编译期的整数序列。  
1. 核心定义

In [ ]:
// 标准定义
template<typename T, T... Ints>
struct integer_sequence;

// 由于我们最常用的整数序列通常是 size_t 类型的索引，标准库专门提供了一个更常用的快捷别名：
// 别名定义：
template<size_t... Ints>
using index_sequence = integer_sequence<size_t, Ints...>;

// 辅助工厂模板（核心：自动生成 0 到 N-1 的序列）：
template<size_t N>
using make_index_sequence = __make_integer_seq<index_sequence, size_t, N>; 
// 注：make_index_sequence<4> 会在编译期自动推导生成：index_sequence<0, 1, 2, 3>


2. Use Cases (The most classic scene: expand std::tuple)

[code](./I_S/integer_sequence_main.cpp)
3. Common Modern C++ Application Scenes  
除了手动展开 tuple，这个工具在各种现代高性能库（如多线程、RPC 序列化、反射库）中无处不在：  
- 将数组/元组作为参数传递给普通函数：
假设你有一个 std::array<int, 3> 或者是 std::tuple，你想把它里面的元素拆开，作为参数传给一个普通函数 void foo(int, int, int)。

In [ ]:
template <typename Func, typename Tuple, size_t... Is>
auto call_impl(Func f, Tuple&& t, std::index_sequence<Is...>) {
    return f(std::get<Is>(std::forward<Tuple>(t))...);
}
// 注：这正是 C++17 标准库标准函数 std::apply 的底层原理。


- 编译期快速生成/转换数组：
比如你想在编译期把一个包含 0, 1, 2, 3 的 std::array 里面的每个元素都乘以 2。

In [ ]:
template <size_t... Is>
constexpr auto create_doubled_array(std::index_sequence<Is...>) {
    return std::array<int, sizeof...(Is)>{ (static_cast<int>(Is) * 2)... };
}
constexpr auto my_arr = create_doubled_array(std::make_index_sequence<5>{});
// my_arr 在编译期直接等于 {0, 2, 4, 6, 8}
